In [1]:
import os
import sys
from pathlib import Path

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("otto-baselines")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

from src.baselines import build_covisit_buys, popular_items, recommend
from src.metrics import recall_at_20

print("ready:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 11:32:35 WARN Utils: Your hostname, Omkars-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.75 instead (on interface en0)
26/11/25 11:32:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 11:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 11:32:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


ready: 4.3.0


In [2]:
from pyspark.sql import functions as F

tr = spark.read.parquet(str(DATA_PROCESSED / "split_train"))
ti = spark.read.parquet(str(DATA_PROCESSED / "test_input"))
tl = spark.read.parquet(str(DATA_PROCESSED / "test_labels"))

max_ts = spark.read.parquet(str(DATA_PROCESSED / "train_events")).agg(F.max("ts")).first()[0]
cutoff = max_ts - 7 * 24 * 3600 + 1

print("cutoff:", cutoff)
print("train events:", tr.count())
print("test sessions:", tl.count())

cutoff: 1661119200
train events: 163955181
test sessions: 1801251


In [3]:
popular = popular_items(tr, cutoff)
cv = build_covisit_buys(tr)
sessions = tl.select("session")

results = {
    "popular only": recall_at_20(recommend(ti, sessions, popular, use_history=False), tl),
    "history + popular": recall_at_20(recommend(ti, sessions, popular), tl),
    "history + covisit + popular": recall_at_20(recommend(ti, sessions, popular, covisit=cv), tl),
}

import pandas as pd
pd.DataFrame(results).T.round(4)

,clicks,carts,orders,score
popular only,0.0053,0.0067,0.0060,0.0061
history + popular,0.3259,0.3122,0.5736,0.4704
history + covisit + popular,0.4244,0.3831,0.6170,0.5276


In [6]:
seqs = (
    tr.groupBy("session")
    .agg(F.sort_array(F.collect_list(F.struct("ts", "aid"))).alias("s"))
    .select("session", F.transform("s", lambda x: x["aid"]).alias("all_items"))
    .withColumn(
        "dedup",
        F.expr("filter(all_items, (x, i) -> i = 0 OR x != all_items[i - 1])"),
    )
    .select(
        "session",
        F.expr("slice(dedup, greatest(1, size(dedup) - 19), 20)").alias("items"),
    )
    .filter(F.size("items") >= 2)
)

seqs.write.mode("overwrite").parquet(str(DATA_PROCESSED / "train_sequences"))

sq = spark.read.parquet(str(DATA_PROCESSED / "train_sequences"))
print("sequences:", sq.count())
sq.select(F.size("items").alias("len")).summary("mean", "50%", "max").show()
sq.show(3, truncate=100)

sequences: 9716693


+-------+-----------------+
|summary|              len|
+-------+-----------------+
|   mean|8.248530955953841|
|    50%|                5|
|    max|               20|
+-------+-----------------+

+-------+----------------------------------------------------------------------------------------------------+
|session|                                                                                               items|
+-------+----------------------------------------------------------------------------------------------------+
|     98|[593662, 932046, 1402537, 932046, 1673641, 932046, 24318, 932046, 1657142, 932046, 990324, 454744...|
|    195|                                                          [485296, 1187885, 706566, 1007308, 376552]|
|    256|[271070, 731352, 1357776, 718941, 731352, 1355234, 1417794, 1308169, 1461739, 1168475, 818093, 16...|
+-------+----------------------------------------------------------------------------------------------------+
only showing top 3 rows


In [7]:
import numpy as np
import pyarrow.parquet as pq

L = 20

items = pq.read_table(DATA_PROCESSED / "train_sequences", columns=["items"]).column("items").combine_chunks()
offsets = items.offsets.to_numpy()
values = items.values.to_numpy().astype(np.int32) + 1  # shift IDs so 0 = padding
lengths = np.diff(offsets)

n = len(lengths)
X = np.zeros((n, L), dtype=np.int32)

rows = np.repeat(np.arange(n), lengths)
pos_in_seq = np.arange(len(values)) - np.repeat(offsets[:-1], lengths)
cols = pos_in_seq + (L - np.repeat(lengths, lengths))  # left padding
X[rows, cols] = values

np.save(DATA_PROCESSED / "train_seqs.npy", X)

print("shape:", X.shape, "| size:", round(X.nbytes / 1e9, 2), "GB")
print("largest ID:", X.max(), "| padding share:", round((X == 0).mean(), 3))
print(X[:3])

shape: (9716693, 20) | size: 0.78 GB
largest ID: 1855603 | padding share: 0.588
[[      0       0       0       0       0       0       0       0       0
        0       0       0       0       0  345694   33119  208147  360814
  1468794 1269913]
 [ 414414 1323310 1777316  734179  543396  368762  543396 1594166 1563669
   250385 1563669 1240414  654314 1240414  238805  908466 1240414  250385
  1230872 1509996]
 [1332590 1671548 1332590  339058  461212   29736 1436281  832193  373290
     9123 1706900 1227358  832193 1469979 1838330  758782  832193  137515
   832193 1436281]]
